# Tutorial: RGB, band-ratio, and nearest-minimum maps with pyFRESCO

This notebook shows how to create and save the new RGB/BW maps available in pyFRESCO:

- standard Viviano-Beck-style RGB maps;
- single-parameter BW maps;
- BW band-ratio maps;
- RGB band-ratio maps;
- nearest-minimum wavelength maps.

The examples use the CRISM observation `FRT00009b5a`. Update the paths below if your data are stored elsewhere.

**Documentation note.** The example figures used by this notebook are expected to be stored in a folder named `notebooks_6/` next to the notebook, following the same structure used by the other pyFRESCO tutorial notebooks.


## Import pyFRESCO

In [ ]:
import pyfresco

## Load the CRISM datacubes

The IF datacube contains the hyperspectral observation. The SR datacube contains the summary parameter products used by several RGB/BW map functions.

In [ ]:
DATA_FOLDER = "notebooks/FRT00009b5a"

path_if_mtrdr = DATA_FOLDER + "/frt00009b5a_07_if165j_mtr3.img"
head_if_mtrdr = DATA_FOLDER + "/frt00009b5a_07_if165j_mtr3.hdr"

path_sr_mtrdr = DATA_FOLDER + "/frt00009b5a_07_sr165j_mtr3.img"
head_sr_mtrdr = DATA_FOLDER + "/frt00009b5a_07_sr165j_mtr3.hdr"

img, img_sr, wavelength, sr_names = pyfresco.open_raw(path_if_mtrdr, head_if_mtrdr, path_sr_mtrdr, head_sr_mtrdr)

Nbands = len(wavelength)
print(sr_names)

## Create a standard false-color RGB map

Here we create the `FAL` RGB map and save it as TXT channels, GeoTIFF, and PNG preview.

In [ ]:
FAL = pyfresco.RGBImageManipulator(img, img_sr, 'FAL', 0, 0, 0, wavelength)

FALmap, selected_stretch = FAL.RGBmapmake(FALSE=FAL, bins=100, clip=True, cumhist=False, preset_true_colors=False, use_false_color=False)

print('Red channel limits = ', '[', selected_stretch[0], ',', selected_stretch[3], ']')
print('Green channel limits = ', '[', selected_stretch[1], ',', selected_stretch[4], ']')
print('Blue channel limits = ', '[', selected_stretch[2], ',', selected_stretch[5], ']')

FAL.savemap('FAL', folder=DATA_FOLDER, map_type='rgb', array=FALmap, save_png=True, show=True)

The output of the interactive stretching cell should look like this:

![FAL false-color RGB map](notebooks_6/FAL_MAKING.png)


## Create a summary-parameter BW map

This example creates a BW map from the `BD1900R2` summary parameter. The false-color map is used as the background reference during interactive stretching.

In [ ]:
BD1900r2 = pyfresco.RGBImageManipulator(img, img_sr, 'BD1900r2', 0, 0, 0, wavelength)

BWmap, stretch_BW = BD1900r2.BWmapmake(bins=1000, sp_param='BD1900R2', FALSE=FALmap, clip=True, cumhist=False, preset_true_colors=None, use_false_color=True, W_min_in=[0, 0.1], W_max_in=[0, 0.1], init_W=[0, 0.1], slider_step=0.0001, slider_height=0.02, slider_width=0.25, slider_spacing=0.05)

print('Black and White channel limits = ', '[', stretch_BW[0], ',', stretch_BW[1], ']')

BD1900r2.savemap('BD1900r2', folder=DATA_FOLDER, map_type='bw', array=BWmap, save_png=True, show=True)

The output of the BW summary-parameter map should look like this:

![BD1900R2 BW map](notebooks_6/BW_MAKING.png)


## Create a stretched BW band-ratio map

This map is computed as reflectance at 1900 nm divided by reflectance at 1500 nm. With `stretch=True`, the output is optimized for display.

In [ ]:
BR_BW, stretch_BW = BD1900r2.BR_BWmapmake(numerator_wavelength=1900, denominator_wavelength=1500, bins=500, stretch=True)

BD1900r2.savemap('band_ratio_1900_1500_map_stretch', folder=DATA_FOLDER, map_type='bw', array=BR_BW, save_png=True, show=True)

The stretched BW band-ratio map should look like this:

![Stretched BW band-ratio map](notebooks_6/BandRatio_MAKING.png)


## Create a raw BW band-ratio map

With `stretch=False`, the output contains the raw band-ratio values. This version is better for quantitative analysis.

In [ ]:
BR_BW_raw, _ = BD1900r2.BR_BWmapmake(numerator_wavelength=1900, denominator_wavelength=1500, bins=500, stretch=False)

BD1900r2.savemap('band_ratio_1900_1500_map_nostretch', folder=DATA_FOLDER, map_type='bw', array=BR_BW_raw, save_png=True, show=True)

The raw BW band-ratio map should look like this:

![Raw BW band-ratio map](notebooks_6/BandRatio_MAKING_NOSTRETCH.png)


## Create a stretched RGB band-ratio map

Each RGB channel is a different band ratio. With `stretch=True`, each channel is stretched for visualization.

In [ ]:
BR_RGB, stretch_RGB = BD1900r2.BR_RGBmapmake(numerator_wavelengths=[2130, 1800, 750], denominator_wavelengths=[2200, 1930, 1000], bins=500, stretch=True)

BD1900r2.savemap('band_ratio_rgb_2130_1800_750_2200_1930_1000_map_stretch', folder=DATA_FOLDER, map_type='rgb', array=BR_RGB, save_png=True, show=True)

The stretched RGB band-ratio map should look like this:

![Stretched RGB band-ratio map](notebooks_6/BandRatio_MAKING_RGB.png)


## Create a raw RGB band-ratio map

With `stretch=False`, the three RGB channels contain raw band-ratio values.

In [ ]:
BR_RGB_raw, _ = BD1900r2.BR_RGBmapmake(numerator_wavelengths=[1900, 2300, 2500], denominator_wavelengths=[1500, 1800, 2000], bins=500, stretch=False)

BD1900r2.savemap('band_ratio_rgb_1900_2300_2500_1500_1800_2000_map_nostretch', folder=DATA_FOLDER, map_type='rgb', array=BR_RGB_raw, save_png=True, show=True)

The raw RGB band-ratio map should look like this:

![Raw RGB band-ratio map](notebooks_6/BandRatio_MAKING_RGB_NOSTRETCH.png)


## Create a nearest-minimum wavelength map

The nearest-minimum map searches, for each pixel, the wavelength of the local minimum closest to a target wavelength. In this example, the target is 1900 nm and the search window is ±100 nm.

The output `NM1900` is the wavelength map in nm. The output `NM1900_value` is the reflectance value at the detected minimum.

In [ ]:
NM = pyfresco.RGBImageManipulator(img, img_sr, None, 0, 0, 0, wavelength)

NM1900, NM1900_value = NM.NM_BWmapmake(target_wavelength=1900, search_window_nm=100, smooth=True, t='savgol', w=7, o=2, colors=("cyan", "black", "orange"))

The nearest-minimum wavelength map should look like this:

![Nearest-minimum wavelength map](notebooks_6/NM_MAKING.png)


## Save the nearest-minimum wavelength map

`save_NMmap()` saves the wavelength map as raw scientific data and, optionally, as a colored GeoTIFF/PNG. Pass `array=NM1900`, not `NM1900_value`, when saving the wavelength map.

In [ ]:
NM.save_NMmap('NM1900_nearest_minimum_nm', folder=DATA_FOLDER, array=NM1900, colors=("cyan", "black", "orange"), save_txt=True, save_raw=True, save_colored=True, save_png=True, show=True)

## Optional: save the minimum-reflectance value map

The `NM1900_value` array is not a wavelength map. It contains the reflectance value measured at the detected minimum. Save it separately as a BW map if needed.

In [ ]:
NM.savemap('NM1900_minimum_reflectance_value', folder=DATA_FOLDER, map_type='bw', array=NM1900_value, save_png=True, show=True)